In [65]:
import numpy as np
import talib
import plotly.graph_objects as go
import plotly.subplots as sub
import yfinance

md = yfinance.download(
    "AAPL",
    start="2017-09-01",
    end="2018-05-10",
    interval="1d",
    progress=False,
)

# md = yfinance.download(
#     "FXI",
#     start="2020-05-01",
#     end="2020-10-16",
#     interval="1D",
#     progress=False,
# )

price = go.Candlestick(
    x=md.index,
    open=md['Open'],
    high=md['High'],
    low=md['Low'],
    close=md['Close'],
    hoverinfo='skip',
    increasing=go.candlestick.Increasing(
        line=dict(width=1),
        fillcolor='#0a9',
        line_color='#0a9',
    ),
    decreasing=go.candlestick.Decreasing(
        line=dict(width=1),
        fillcolor='#f44',
        line_color='#f44',
    )
)

layout = go.Layout(
    # template='plotly_white',
    width=800,
    height=500,
    autosize=False,
    margin=dict(t=30, b=10, l=60, r=30),
    showlegend=False,
    paper_bgcolor='#f5f5f5',
    plot_bgcolor='#fff',
    xaxis=dict(
        linecolor="#ddd",
        # showspikes=True,
        # # Format spike
        # spikethickness=1,
        # spikedash="dot",
        # spikecolor="#999999",
        # spikemode="across",
        # grid
        showgrid=True,
        gridcolor="#ddd",
        fixedrange=True,
    ),
    yaxis=dict(
        linecolor="#ddd",
        # grid
        showgrid=True,
        gridcolor="#ddd",
        fixedrange=True,
    )
)

In [88]:
fig = sub.make_subplots(
    figure=go.Figure(layout=layout),
    rows=2,
    cols=1,
    shared_xaxes=True,
    shared_yaxes=False,
    vertical_spacing=0.05,
    row_heights=[0.75, 0.25],
    subplot_titles=["Price", "Indicators"],
)

fig.add_trace(price, 1, 1)

# Parabolic SAR
# md["indi"] = talib.SAR(md["High"], md["Low"], acceleration=0.03, maximum=0.2)

# md["indi"] = talib.KAMA(md["Close"])
# md["trix"] = talib.TRIX(md["Close"], timeperiod=10)

# ATR
md["atr"] = talib.ATR(md["High"], md["Low"], md["Close"], timeperiod=5)

md["ema1"] = talib.MA(md["Close"], timeperiod=14, matype=1)
md["ema2"] = talib.MA(md["Close"], timeperiod=14, matype=0)
md["ema"] = (md["ema1"] + md["ema2"]) / 2

md["stdev"] = talib.STDDEV(md["Close"], timeperiod=20, nbdev=1)

ind = md[["atr", "stdev"]].max(axis=1)

md["top"] = md["ema"] + 2 * ind
md["bottom"] = md["ema"] - 2 * ind



line_m = dict(width=2, color='#59c')
fig.add_trace(go.Scatter(x=md.index, y=md["ema"], fill=None, line=line_m))

# line_m = dict(width=2, color='#666')
# mmm = md["ema"] + 2 * md["atr"]
# fig.add_trace(go.Scatter(x=md.index, y=mmm, fill=None, line=line_m))


# aroondown, aroonup = talib.AROON(md["High"], md["Low"], timeperiod=14)
# macd, macdsignal, macdhist = talib.MACD(md["Close"], fastperiod=12, slowperiod=26, signalperiod=9)
# ind = macd
# ind = talib.STDDEV(md["Close"], timeperiod=10, nbdev=1)


# Вот это похоже на RANGE STRENGTH
# ind = talib.CCI(md["High"], md["Low"], md["Close"], timeperiod=40)
# ind = talib.DX(md["High"], md["Low"], md["Close"], timeperiod=50)


ind = 24 - talib.ADX(md["High"], md["Low"], md["Close"], timeperiod=14)
ind_smooth = talib.MA(ind, timeperiod=2, matype=0)


ind_color = np.where(ind_smooth < 0, 'red', '#0c0')


line_m = dict(width=2, color='#f50')
# fig.add_trace(go.Scatter(x=md.index, y=ind_smooth, fill=None, line=line_m), row=2, col=1)
bars = go.Bar(x=md.index, y=ind_smooth, marker_color=ind_color, width=40000000)
fig.add_trace(bars, row=2, col=1)


# BB
# bb_u, bb_m, bb_l = talib.BBANDS(md['Close'], timeperiod=10, nbdevup=3, nbdevdn=3)
# line = dict(width=0.5, color='#9b9')
# line_m = dict(width=2, color='#59c')
# fig.add_trace(go.Scatter(x=md.index, y=bb_m, fill=None, line=line_m))
# fig.add_trace(go.Scatter(x=md.index, y=bb_u, fill=None, line=line))
# fig.add_trace(go.Scatter(x=md.index, y=bb_l, fill='tonexty', line=line))


# СИГНАЛЫ
sig = talib.CDLHAMMER(
    open=md['Open'],
    high=md['High'],
    low=md['Low'],
    close=md['Close'],
)
annotations = []
for dt, value in sig.items():
    if value > 0:
        annotations.append({
            "x": dt,
            "y": md.loc[dt]["Close"],
            "text": len(annotations) + 1,
        })
fig.update_layout(annotations=annotations)


# Убрать дыры в оси X
fig.update_xaxes(
    range=['2017-12-03 09:30:00-04:00', '2018-05-03 11:30:00-04:00'],
    # range=['2020-06-16 09:30:00-04:00', '2020-10-16 11:30:00-04:00'],
    rangeslider_visible=False,
    rangebreaks=[
        dict(bounds=["sat", "mon"]),  # hide weekends
        # dict(bounds=[16, 9.5], pattern="hour"),  # hide extra hours
        dict(values=[
            "2017-12-25",
            "2018-01-01",
            "2018-01-15",
            "2018-02-19",
            "2018-03-30",
        ])  # hide holidays
    ]
)

fig.update_yaxes(
    fixedrange=True,
    # range=[33, 48],
    row=1,
)


# Индикатор на графике цены
indicator = go.Scatter(
    x=md.index,
    y=md["top"],
    mode="lines",
    line=dict(color="red", width=2),
)
fig.add_trace(indicator, 1, 1)
indicator = go.Scatter(
    x=md.index,
    y=md["bottom"],
    mode="lines",
    line=dict(color="#0c0", width=2),
)
fig.add_trace(indicator, 1, 1)

# BAR
# bars = go.Bar(x)

# Индикатор на отдельном графике
# indicator = go.Scatter(
#     x=md.index,
#     y=ind,
#     mode="lines",
#     line=dict(color="blue", width=1),
#     marker=dict(color="black", size=2),
# )
# fig.add_trace(indicator, 2, 1)
# fig.update_yaxes(
#     fixedrange=True,
#     # range=[0, 2],
#     row=2,
# )
# Оформление индикатора
fig.add_hline(y=0, line=dict(color="#666", width=1), row=2)
# fig.add_hline(y=0.3, line=dict(color="blue", width=1), row=2)
# fig.add_hline(y=-0.3, line=dict(color="red", width=1), row=2)
# fig.add_hrect(y0=0, y1=20, line_width=0, fillcolor="green", opacity=0.1, row=2)
# fig.add_hrect(y0=0, y1=-20, line_width=0, fillcolor="red", opacity=0.1, row=2)

fig.show(config={"displayModeBar": False, "showTips": False})
